In [1]:
# ===== Cell 1: 封装一个能执行终端命令的 Langchain 工具 =====
# 为什么不用官方的 ShellTool？
#   1) 教程里写的 from langchain_tools import ShellTool 这个包根本不存在（会报 ModuleNotFoundError）
#      正确路径本来是 langchain_community.tools.shell
#   2) 但官方 ShellTool 底层执行的是 bash（Linux 命令），相关代码已移到 langchain_experimental 包，
#      而 Windows 默认没有 bash 环境 —— 所以在本机怎么装都跑不通。
#
# 解决办法：用 Python 内置的 subprocess（专门用来启动系统进程/执行命令的库）
#           + @tool 装饰器，自己做一个 Windows 上原生可用的 shell 工具。
import subprocess
from langchain_core.tools import tool


@tool
def shell_tool(command: str) -> str:
    '''在 Windows 终端(cmd)中执行一条命令并返回输出结果。
    当用户要求查看目录、列出文件、查询系统信息、运行命令行指令时使用此工具。

    参数:
        command: 要执行的命令，例如 "dir"、"echo hello"、"cd"
    '''
    try:
        # 中文乱码的坑（重点）：
        #   Windows 中文版 cmd 默认是 GBK 代码页，直接读取管道经常把中文路径/文件名弄乱码。
        #   解决：命令前加 "chcp 65001 >nul &&" 先把当前终端切到 UTF-8 代码页，
        #         然后用 encoding='utf-8' 解码输出，中文就能正常显示。
        full_command = f'chcp 65001 >nul && {command}'
        # shell=True  : 让命令通过系统的 cmd.exe 解释执行，这样 dir、echo 等内置命令才能用
        # capture_output=True : 捕获命令的正常输出和报错信息，而不是直接打印到屏幕
        # text=True  : 把输出按文本(字符串)返回，而不是字节
        # timeout=30 : 防止执行到死循环或卡住的命令，最多等 30 秒
        result = subprocess.run(
            full_command,
            shell=True,
            capture_output=True,
            text=True,
            encoding='utf-8',
            errors='replace',
            timeout=30,
        )
        output = result.stdout.strip()
        error = result.stderr.strip()
        if result.returncode != 0:
            # returncode 非 0 表示命令执行出错
            return f'命令执行失败(返回码 {result.returncode}):\n{error}'
        return output if output else '(命令执行成功，但没有输出内容)'
    except subprocess.TimeoutExpired:
        return '命令执行超时（超过30秒），已被终止。'
    except Exception as e:
        return f'执行命令时发生异常: {e}'


print('Windows Shell 工具封装完成')
print(f'工具名: {shell_tool.name}')
print(f'工具说明: {shell_tool.description}')



Windows Shell 工具封装完成
工具名: shell_tool
工具说明: 在 Windows 终端(cmd)中执行一条命令并返回输出结果。
    当用户要求查看目录、列出文件、查询系统信息、运行命令行指令时使用此工具。

    参数:
        command: 要执行的命令，例如 "dir"、"echo hello"、"cd"


In [2]:
# shell_tool.invoke({'command': 'start msedge www.baidu.com'})

In [3]:
# ===== Cell 2: 直接调用 Shell 工具，测试常见命令 =====
# invoke 的参数要传字典，key 必须和函数参数名(command)一致

print('--- 测试1: echo 输出文本 ---')
print(shell_tool.invoke({'command': 'echo hello world'}))

print('\n--- 测试2: ver 查看系统版本 ---')
print(shell_tool.invoke({'command': 'ver'}))

print('\n--- 测试3: cd 查看当前工作目录 ---')
print(shell_tool.invoke({'command': 'cd'}))

print('\n--- 测试4: dir 列出当前目录文件 ---')
print(shell_tool.invoke({'command': 'dir'})[:400])  # 文件多，只打印前400字

print('\n--- 测试5: 故意执行一条错误命令，看错误处理 ---')
print(shell_tool.invoke({'command': 'this_is_not_a_command_xyz'}))



--- 测试1: echo 输出文本 ---
hello world

--- 测试2: ver 查看系统版本 ---
Microsoft Windows [Version 10.0.22631.4751]

--- 测试3: cd 查看当前工作目录 ---
S:\PythonAILearningCourse\07-langchain-rag

--- 测试4: dir 列出当前目录文件 ---
Volume in drive S is 新加卷
 Volume Serial Number is 7CD2-8E70

 Directory of S:\PythonAILearningCourse\07-langchain-rag

2026/09/13  02:11    <DIR>          .
2026/09/04  19:16    <DIR>          ..
2026/09/13  01:18    <DIR>          .ipynb_checkpoints
2026/09/10  17:49             9,237 .~05.RAG问答系统.ipynb
2026/09/10  22:12            54,645 01.Langchain.ipynb
2026/09/07  15:30            15,880 02.向量存储和数据检

--- 测试5: 故意执行一条错误命令，看错误处理 ---
命令执行失败(返回码 1):
'this_is_not_a_command_xyz' is not recognized as an internal or external command,
operable program or batch file.


In [4]:
# ===== Cell 3: 接入 Agent，让大模型自动调用终端命令 =====
# 和 05 课的天气 Agent 同样的套路：
#   用户提问 -> 大模型判断需要执行什么命令 -> 自动调用 shell_tool -> 根据真实输出回答
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

llm = ChatOpenAI(
    model='sensenova-6.8-flash-lite',
    base_url='https://token.sensenova.cn/v1',
    api_key=os.getenv('SENSENOVA_API_KEY', ''),
    temperature=0.1,  # 执行命令要严谨，温度调低
)

# 安全提醒写进系统提示词：只执行查询类的安全命令
agent = create_agent(
    model=llm,
    tools=[shell_tool],
    system_prompt=(
        '你是一个 Windows 系统助手。当需要了解系统或文件信息时，可以调用 shell_tool 执行命令。'
        '允许运行start msedge www.baidu.com类似打开百度网页的命令'
        '只允许执行 dir、cd、ver、echo、ipconfig、whoami 等只读的安全查询命令；'
        '严禁执行删除文件、格式化、修改系统设置等任何有破坏性的命令。'
        '拿到命令输出后，用中文简洁地总结给用户。'
    ),
)
print('命令行 Agent 创建完成')



命令行 Agent 创建完成


In [7]:
# agent.invoke('帮我打开百度')

InvalidUpdateError: Expected dict, got 帮我打开百度
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_GRAPH_NODE_RETURN_VALUE

In [5]:
# ===== Cell 4: 运行 Agent，观察它自动执行终端命令 =====
def chat(user_input):
    result = agent.invoke({
        'messages': [{'role': 'user', 'content': user_input}]
    })
    answer = result['messages'][-1].content.strip()
    print(f'我: {user_input}')
    print(f'助手: {answer}\n')
    # 返回完整过程，方便下一步查看它执行了什么命令
    return result

# 让 Agent 自己决定该执行哪条命令
r1 = chat('帮我看看当前在哪个目录工作？')
r2 = chat('这个文件夹里都有哪些文件？')



我: 帮我看看当前在哪个目录工作？
助手: 当前工作目录是：

**`S:\PythonAILearningCourse\07-langchain-rag`**

看起来你正在一个关于 LangChain 的 Python 人工智能课程目录下工作。需要我帮你查看这个目录下的文件吗？

我: 这个文件夹里都有哪些文件？
助手: 当前文件夹 `S:\PythonAILearningCourse\07-langchain-rag` 的内容如下：

**📁 子文件夹（4 个）**
- `.ipynb_checkpoints`（Jupyter 自动备份）
- `txt`

**📄 文件（11 个，共约 180 KB）**

| 文件名 | 大小 |
|---|---|
| 01.Langchain.ipynb | 54,645 B |
| 02.向量存储和数据检索.ipynb | 15,880 B |
| 03.文档加载器.ipynb | 5,326 B |
| 04.a付费模型接入Langchain.ipynb | 13,124 B |
| 04.RAG问答系统.ipynb | 14,367 B |
| 05.Langchain调用工具.ipynb | 16,461 B |
| 06.fess文档搜索服务器.ipynb | 6,731 B |
| 06.fess，api接口搜索.ipynb | 2,465 B |
| 07.调用shell执行终端命令.ipynb | 25,926 B |
| .~05.RAG问答系统.ipynb（临时文件） | 9,237 B |
| article_333623112.txt | 16,112 B |

整体来看，这是一个 **LangChain 学习课程** 的笔记目录，包含从基础概念、向量存储、文档加载、RAG 问答系统、工具调用到 shell 命令执行的完整系列 Jupyter Notebook。

💡 小提示：`.~05.RAG问答系统.ipynb` 是 Jupyter 的临时锁文件，通常可以安全删除；另外有两个 `04.` 和两个 `06.` 开头的文件，命名上有点重复，可能需要整理一下。



In [6]:
# ===== Cell 5: 查看 Agent 的完整动作（执行了哪条命令）=====
for msg in r1['messages']:
    if msg.type == 'human':
        print(f'[用户] {msg.content}')
    elif msg.type == 'ai' and getattr(msg, 'tool_calls', None):
        for call in msg.tool_calls:
            print(f'[大模型决策] 执行命令: {call["args"].get("command")}')
    elif msg.type == 'tool':
        print(f'[终端输出] {msg.content[:200]}')
    elif msg.type == 'ai':
        print(f'[大模型回答] {msg.content.strip()}')



[用户] 帮我看看当前在哪个目录工作？
[大模型决策] 执行命令: cd
[终端输出] S:\PythonAILearningCourse\07-langchain-rag
[大模型回答] 当前工作目录是：

**`S:\PythonAILearningCourse\07-langchain-rag`**

看起来你正在一个关于 LangChain 的 Python 人工智能课程目录下工作。需要我帮你查看这个目录下的文件吗？


In [7]:
chat('帮我打开百度')

我: 帮我打开百度
助手: 已经帮你用 Microsoft Edge 浏览器打开了百度（www.baidu.com）🌐

如果浏览器没有自动弹出，可能是被系统拦截了，可以告诉我，我再帮你换个方式打开。



{'messages': [HumanMessage(content='帮我打开百度', additional_kwargs={}, response_metadata={}, id='863db74c-a715-4530-9293-6bad18fe2299'),
  AIMessage(content='\n\n', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 501, 'total_tokens': 578, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 43, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'sensenova-6.8-flash-lite', 'system_fingerprint': None, 'id': '11e3fb1e-1e80-4233-9f57-011bbf2fb6eb', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a096d4-0c3d-7d21-b7c6-8957fd42dc47-0', tool_calls=[{'name': 'shell_tool', 'args': {'command': 'start msedge www.baidu.com'}, 'id': 'call_d6d993505de648d5ba550880', 'type': 'tool_call'